In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [10, 6]

DATA_DIR = Path("../../data/1_interim")

df_students = pd.read_csv(DATA_DIR / "students.csv")
df_assessments = pd.read_csv(DATA_DIR / "assessments.csv")
df_interactions = pd.read_csv(DATA_DIR / "interactions.csv")

print("Datos cargados correctamente:")
print(f"Estudiantes: {df_students.shape}")
print(f"Evaluaciones: {df_assessments.shape}")
print(f"Interacciones: {df_interactions.shape}")

Datos cargados correctamente:
Estudiantes: (32593, 15)
Evaluaciones: (173912, 10)
Interacciones: (10655280, 9)


In [2]:
display(df_students.head(5))

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,date_registration,date_unregistration,module_presentation_length
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,-159.0,NaN,268
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass,-53.0,NaN,268
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn,-92.0,12.0,268
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass,-52.0,NaN,268
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass,-176.0,NaN,268


In [3]:
columnas_es = {
    "code_module": "Código\nmódulo",
    "code_presentation": "Código\npresentación",
    "id_student": "ID\nestudiante",
    "gender": "Género",
    "region": "Región",
    "highest_education": "Nivel\neducativo",
    "imd_band": "Banda\nIMD",
    "age_band": "Rango de\nedad",
    "num_of_prev_attempts": "Número de\nintentos previos",
    "studied_credits": "Créditos\ncursados",
    "disability": "Discapacidad",
    "final_result": "Clase\ntarget",
    "date_registration": "Día de\nregistro",
    "date_unregistration": "Día de\nbaja",
    "module_presentation_length": "Duración\nmódulo-presentación",
}


def preparar_tabla(
    df,
    columnas_map,
    columnas_dias=None,
    columnas_id_compuesto=None,
    nombre_id_compuesto="ID\ncompuesto",
    columnas_sin_decimales=None,
    columna_final=None,
    columnas_eliminar=None,
):
    df_es = df.rename(columns=columnas_map).copy()

    for columna in columnas_dias or []:
        nombre_columna = columnas_map.get(columna, columna)
        df_es[nombre_columna] = pd.to_numeric(
            df_es[nombre_columna], errors="coerce"
        ).astype("Int64")

    for columna in columnas_sin_decimales or []:
        nombre_columna = columnas_map.get(columna, columna)
        df_es[nombre_columna] = pd.to_numeric(
            df_es[nombre_columna], errors="coerce"
        ).map(lambda valor: f"{valor:g}" if pd.notna(valor) else pd.NA)

    if columnas_id_compuesto:
        nombres_id = [columnas_map.get(columna, columna) for columna in columnas_id_compuesto]
        df_es.insert(
            0,
            nombre_id_compuesto,
            df_es[nombres_id].astype(str).agg("_".join, axis=1),
        )
        df_es = df_es.drop(columns=nombres_id)

    if columnas_eliminar:
        nombres_columnas_eliminar = [
            columnas_map.get(columna, columna) for columna in columnas_eliminar
        ]
        df_es = df_es.drop(columns=nombres_columnas_eliminar, errors="ignore")

    if columna_final:
        nombre_columna_final = columnas_map.get(columna_final, columna_final)
        columnas_ordenadas = [
            columna for columna in df_es.columns if columna != nombre_columna_final
        ] + [nombre_columna_final]
        df_es = df_es[columnas_ordenadas]

    return df_es



def mostrar_tabla(df):
    display(
        df.head(10).style.set_table_styles(
            [
                {
                    "selector": "th.col_heading",
                    "props": [("white-space", "pre-line")],
                }
            ]
        )
    )



df_students_es = preparar_tabla(
    df_students,
    columnas_es,
    ["date_registration", "date_unregistration"],
    ["code_module", "code_presentation", "id_student"],
    columna_final="final_result",
    columnas_eliminar=["gender", "disability"],
)

mostrar_tabla(df_students_es)

,ID compuesto,Región,Nivel educativo,Banda IMD,Rango de edad,Número de intentos previos,Créditos cursados,Día de registro,Día de baja,Duración módulo-presentación,Clase target
0,AAA_2013J_11391,East Anglian Region,HE Qualification,90-100%,55<=,0,240,-159,,268,Pass
1,AAA_2013J_28400,Scotland,HE Qualification,20-30%,35-55,0,60,-53,,268,Pass
2,AAA_2013J_30268,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,-92,12,268,Withdrawn
3,AAA_2013J_31604,South East Region,A Level or Equivalent,50-60%,35-55,0,60,-52,,268,Pass
4,AAA_2013J_32885,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,-176,,268,Pass
5,AAA_2013J_38053,Wales,A Level or Equivalent,80-90%,35-55,0,60,-110,,268,Pass
6,AAA_2013J_45462,Scotland,HE Qualification,30-40%,0-35,0,60,-67,,268,Pass
7,AAA_2013J_45642,North Western Region,A Level or Equivalent,90-100%,0-35,0,120,-29,,268,Pass
8,AAA_2013J_52130,East Anglian Region,A Level or Equivalent,70-80%,0-35,0,90,-33,,268,Pass
9,AAA_2013J_53025,North Region,Post Graduate Qualification,nan,55<=,0,60,-179,,268,Pass


In [5]:
columnas_assessments_es = {
    "id_assessment": "ID\nevaluación",
    "id_student": "ID\nestudiante",
    "date_submitted": "Día de\nentrega",
    "is_banked": "En\nbanco",
    "score": "Puntuación",
    "code_module": "Código\nmódulo",
    "code_presentation": "Código\npresentación",
    "assessment_type": "Tipo de\nevaluación",
    "date": "Día",
    "weight": "Peso",
}

df_assessments_es = preparar_tabla(
    df_assessments,
    columnas_assessments_es,
    ["date_submitted", "date"],
    ["code_module", "code_presentation", "id_student"],
    columnas_sin_decimales=["weight"],
)

df_assessments_es_muestra = (
    df_assessments_es.sort_values(["Tipo de\nevaluación", "ID\ncompuesto", "Día"])
    .groupby("Tipo de\nevaluación", group_keys=False)
    .head(3)
    .reset_index(drop=True)
)

mostrar_tabla(df_assessments_es_muestra)

columnas_interactions_es = {
    "code_module": "Código\nmódulo",
    "code_presentation": "Código\npresentación",
    "id_student": "ID\nestudiante",
    "id_site": "ID\nsitio",
    "date": "Día",
    "sum_click": "Total de\nclicks",
    "activity_type": "Tipo de\nactividad",
    "week_from": "Semana\ndesde",
    "week_to": "Semana\nhasta",
}

df_interactions_es = preparar_tabla(
    df_interactions,
    columnas_interactions_es,
    ["date"],
    ["code_module", "code_presentation", "id_student"],
)

mostrar_tabla(df_interactions_es)

,ID compuesto,ID evaluación,Día de entrega,En banco,Puntuación,Tipo de evaluación,Día,Peso
0,BBB_2013B_1008675,14991,60,0,40.000000,CMA,54,1
1,BBB_2013B_1008675,14992,95,0,100.000000,CMA,89,1
2,BBB_2013B_1008675,14993,126,0,100.000000,CMA,124,1
3,CCC_2014B_1038161,24290,236,0,58.000000,Exam,,100
4,CCC_2014B_105523,24290,236,0,60.000000,Exam,,100
5,CCC_2014B_1057883,24290,230,0,78.000000,Exam,,100
6,AAA_2013J_100893,1752,17,0,63.000000,TMA,19,10
7,AAA_2013J_100893,1753,50,0,68.000000,TMA,54,20
8,AAA_2013J_100893,1754,116,0,71.000000,TMA,117,20


,ID compuesto,ID sitio,Día,Total de clicks,Tipo de actividad,Semana desde,Semana hasta
0,AAA_2013J_28400,546652,-10,4,forumng,nan,nan
1,AAA_2013J_28400,546652,-10,1,forumng,nan,nan
2,AAA_2013J_28400,546652,-10,1,forumng,nan,nan
3,AAA_2013J_28400,546614,-10,11,homepage,nan,nan
4,AAA_2013J_28400,546714,-10,1,oucontent,nan,nan
5,AAA_2013J_28400,546652,-10,8,forumng,nan,nan
6,AAA_2013J_28400,546876,-10,2,subpage,nan,nan
7,AAA_2013J_28400,546688,-10,15,oucontent,nan,nan
8,AAA_2013J_28400,546662,-10,17,oucontent,nan,nan
9,AAA_2013J_28400,546890,-10,1,url,nan,nan
